<a href="https://colab.research.google.com/github/andrew-veriga/Titans_jax/blob/main/colabs/Titans_jax_Phase2_LM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Gemma-Titans Phase 2: LM Fine-Tuning

Фаза 2 обучения: переключаемся с послойной дистилляции на language modeling loss.

**Отличия от Phase 1:**
- Выходы студента передаются между слоями (нет `stop_gradient`, нет teacher chain)
- Loss: cross-entropy по токенам вместо per-layer MSE
- `training_phase=2` в конфиге модели
- Веса загружаются из `saved_titans_delta` (результат Phase 1)
- Сниженные LR (веса уже предобучены)

In [ ]:
# 0. Environment Setup
!git clone --depth 1 https://github.com/google-research/kauldron || true
!pip install -q ./kauldron
!git clone --depth 1 https://github.com/google-deepmind/gemma.git || true
!pip install -q ./gemma
!git clone --depth 1 https://github.com/google-deepmind/dialog || true
!pip install -q ./dialog
!pip install -q flax optax seqio
!pip install importlib_resources

In [ ]:
!git clone --depth 1 https://github.com/andrew-veriga/Titans_jax.git

In [ ]:
!cp "/content/drive/Shareddrives/shared_veriga/saved_titans_delta/saved_titans_phase2_from_11.zip" saved_titans_delta.zip

## Start

In [ ]:
import os
os._exit(0)

In [ ]:
import sys
import os
sys.path.append(os.getcwd())

import jax
import jax.numpy as jnp
import optax
import dataclasses
import numpy as np
import os
import orbax.checkpoint as ocp
import shutil

from kauldron import kd
from kauldron.ktyping import Array
from gemma import gm
from gemma.gm.nn import _config

# Our custom Titans integration
import importlib

%cd /content/Titans_jax
import gemma_titans
importlib.reload(gemma_titans)
from gemma_titans import Gemma3_1B_Titans, Gemma_Titans_Config
from titans_ckpts import SkipTitans
import titans_tree_utils

print(f"JAX Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")
""" старые настройки
# Prevent JAX from allocating 100% of TPU memory instantly
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# Limit XLA to 85% of TPU HBM to leave room for overhead
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".85"
# Reduce fragmentation and compilation memory spike
os.environ["XLA_FLAGS"] = "--xla_gpu_enable_highest_priority_async_collectives=true --xla_tpu_enable_data_parallel_all_reduce_opt=true --xla_tpu_memory_bound_loop_fusion_limit=1"
os.environ["JAX_COMPILATION_CACHE_DIR"] = "/tmp/jax_cache"
"""
# Разрешаем JAX забрать память сразу для максимальной скорости
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "true"

# Увеличиваем долю памяти (оставляем чуть-чуть на системные нужды)
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".95"

# Оптимизируем флаги для производительности, а не для экономии
os.environ["XLA_FLAGS"] = (
    "--xla_tpu_enable_data_parallel_all_reduce_opt=true "
    "--xla_tpu_joint_all_gather_opt=true "
    "--xla_tpu_enable_latency_hiding_scheduler=true " # Скрывает задержки памяти за вычислениями
    "--xla_tpu_all_reduce_combine_threshold_bytes=134217728" # Оптимально для больших батчей
)

os.environ["JAX_COMPILATION_CACHE_DIR"] = "/tmp/jax_cache"

print(f"JAX Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")

## 2. Гиперпараметры

In [ ]:
batch_size = 16
max_length = 2048
total_steps = 80000

titans_phase2_first_layer = 23  # Titans layers from this index onward are active.
                                 # Earlier layers revert to standard Gemma blocks.
                                 # 17 → layers 17,23 active (~25GB compile RAM)
                                 # 23 → layer 23 only  (~5GB compile RAM)


_all_titans_layers = (11, 17, 23)
active_titans_layers = tuple(l for l in _all_titans_layers if l >= titans_phase2_first_layer)
print(f"Active Titans layers: {active_titans_layers}")


### Расписания Learning Rate для оптимизаторов Titans Memory и параметра delta функции Huber Loss

In [ ]:
# Гиперпараметры
huber_loss_delta = optax.constant_schedule(0.2)
# optax.linear_schedule(
#    init_value=0.1, end_value=.8, transition_begin = 100, transition_steps=total_steps)
# optax.constant_schedule(0.1)
# huber_loss_delta = optax.cosine_decay_schedule(
#     init_value=0.8,
#     decay_steps=10000,
#     alpha=0.1,
# )
  # Можно передать число или optax schedule
# Сниженные LR: веса уже предобучены в Phase 1
# lr_muon = optax.warmup_cosine_decay_schedule(
#     init_value=1e-4,
#     peak_value=5e-4,
#     warmup_steps=500,
#     decay_steps=total_steps - 500,
#     end_value=1e-5,
# )
lr_muon = optax.cosine_decay_schedule(
    init_value=5e-4,
    decay_steps=total_steps,
    alpha=0.1,  # конечный lr = 5e-4 * 0.1 = 5e-4
)
# lr_adam = optax.warmup_cosine_decay_schedule(
#     init_value=1e-4,
#     peak_value=5e-4,
#     warmup_steps=500,
#     decay_steps=total_steps - 500,
#     end_value=1e-5,
# )
lr_adam = optax.cosine_decay_schedule(
    init_value=1e-4,
    decay_steps=total_steps,
    alpha=0.1,# конечный lr = 1e-3 * 0.1 = 1e-4
)

lr_gate = optax.constant_schedule(1.e-3)
#     init_value=1e-4,  # В 100 раз больше, чем lr_adam (1e-5)
#     decay_steps=total_steps,
#     alpha=0.1, # конечный lr = 5e-3 * 0.2 = 1e-4
# )

## 1. Загрузка весов


In [ ]:
def load_titans_weights(load_dir: str):
    checkpointer = ocp.StandardCheckpointer()
    return checkpointer.restore(os.path.abspath(load_dir))

titans_zip = "saved_titans_delta.zip"
titans_delta_path = "./saved_titans_delta"

if os.path.exists(titans_zip) and not os.path.exists(titans_delta_path):
    print(f"Unpacking {titans_zip}...")
    shutil.unpack_archive(titans_zip, titans_delta_path)

print("Loading Gemma base weights...")
original_params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)

print("Loading Phase 1 Titans weights...")
loaded_titans_params = load_titans_weights(titans_delta_path)

# Keep only weights for active Titans layers — inactive layers (< titans_phase2_first_layer)
# revert to original Gemma weights automatically via merge.
active_layer_keys = {f'layer_{l}' for l in active_titans_layers}
loaded_titans_params = {k: v for k, v in loaded_titans_params.items() if k in active_layer_keys}
print(f"Merging Titans weights for: {sorted(active_layer_keys)}")

merged_params = titans_tree_utils.merge_titans_params(original_params, loaded_titans_params, remove_dead_attn=True)
print("✅ Phase 1 weights loaded and merged.")

## 3. Модель (training_phase=2)

In [ ]:

gemma_config = dataclasses.replace(
    Gemma3_1B_Titans.config,
    # sliding_window_size=128,
    training_phase=2,
    titans_layer_indices=active_titans_layers,
    titans_phase2_first_layer=titans_phase2_first_layer,
    neural_mem_huber_delta=huber_loss_delta,
)

model = Gemma3_1B_Titans(
    config=gemma_config,
    dtype=jnp.bfloat16,
    return_last_only=False,
    tokens="batch.tokens",
    # step="step",
)

## 4. Датасет

In [ ]:
"""
Kauldron data loader module for OpenWebText (replaces c4/webtextlike).
"""

import os
import dataclasses
import numpy as np
import grain.python as grain
from kauldron import kd

tokenizer = gm.text.Gemma3Tokenizer()

@dataclasses.dataclass(frozen=True)
class CLMTask(grain.MapTransform):
    """Tokenize full document text → fixed-length tokens array for CLM training.

    Loss is computed on every token position — no src/dst masking.
    Suitable for Wikipedia, PG-19, C4, etc.
    """
    in_text: str
    out_tokens: str
    tokenizer: object
    max_length: int

    def map(self, element):
        text = element[self.in_text]
        if isinstance(text, bytes):
            text = text.decode("utf-8")
        token_ids = self.tokenizer.encode(text, add_bos=True)
        token_ids = token_ids[:self.max_length]
        pad_len = self.max_length - len(token_ids)
        import numpy as np
        token_ids = np.pad(np.array(token_ids, dtype=np.int32), (0, pad_len))
        out = dict(element)
        out[self.out_tokens] = token_ids
        return out

@dataclasses.dataclass(frozen=True)
class TokenizeText(grain.MapTransform):
    """Tokenizes the text field into a fixed-length tokens array."""
    in_text: str = "text"
    out_tokens: str = "tokens"
    tokenizer: object = None
    max_length: int = 1024

    def map(self, element: dict) -> dict:
        # HuggingFace datasets usually provide strings directly, no decoding needed
        text = element[self.in_text]

        # Assuming tokenizer has an encode method (like sentencepiece)
        if hasattr(self.tokenizer, "encode"):
            # Check if add_bos is supported, some tokenizers use it
            try:
                token_ids = self.tokenizer.encode(text, add_bos=True)
            except TypeError:
                token_ids = self.tokenizer.encode(text)
        else:
            token_ids = self.tokenizer(text)

        token_ids = token_ids[:self.max_length]
        pad_len = self.max_length - len(token_ids)
        token_ids = np.pad(np.array(token_ids, dtype=np.int32), (0, pad_len))

        out = dict(element)
        out[self.out_tokens] = token_ids
        return out


def get_c4_webtextlike_dataset(
    split: str = "train",
    batch_size: int = 8,
    tokenizer: object = None,
    max_length: int = 1024,
    data_dir: str = None, # Unused for HuggingFace, kept for API compatibility
    num_epochs: int = None,
    shuffle: bool = True,
):
    """
    Returns a Kauldron dataset pipeline for OpenWebText via HuggingFace.
    This replaces the broken c4/webtextlike TFDS builder.

    Args:
        split: The dataset split to load (e.g., 'train', 'validation').
        batch_size: Number of examples per batch.
        tokenizer: Optional tokenizer object to tokenize the text.
                   If None, raw text is returned.
        max_length: Maximum sequence length for tokenization.
        data_dir: Unused. Kept for API compatibility.
        num_epochs: Number of epochs to iterate (None for infinite).
        shuffle: Whether to shuffle the dataset.

    Returns:
        A Kauldron data source iterator.
    """
    transforms = []

    if tokenizer is not None:
        transforms.append(
            CLMTask(
                in_text="text",
                out_tokens="tokens",
                tokenizer=tokenizer,
                max_length=max_length
            )
        )
        keep_fields = ["tokens"]
    else:
        keep_fields = ["text"]

    transforms.append(kd.data.py.Elements(keep=keep_fields))

    # Using the Kauldron HuggingFace loader to completely bypass Apache Beam
    dataset = kd.data.py.HuggingFace(
        path="Skylion007/openwebtext",
        num_workers=1,
        split=split,
        shuffle=shuffle,
        num_epochs=num_epochs,
        batch_size=batch_size,
        shard_by_process=False,
        transforms=transforms,
        cache_dir='/content/drive/Shareddrives/shared_veriga/tfds_cache',
    )

    return dataset

print("Setting up c4/webtextlike Kauldron pipeline...")
# pipeline = get_c4_webtextlike_dataset(
#     split="train",
#     batch_size=2,
#     tokenizer=None, # pass tokenizer instance here
#     shuffle=False,
#     num_epochs=None
# )

# print("Pipeline ready.")
# print("To yield examples, you would iterate over pipeline(seed=42):")
# # For example:
# it = iter(pipeline)
# batch = next(it)
# print(batch["text"])


## 5. Оптимизатор (сниженный LR для дообучения)

In [ ]:
from adam_atan2 import adam_atan2
from optax.contrib._muon import MuonDimensionNumbers

def muon_only_dims(params):
    MUON_KEYS = {'to_queries', 'to_keys_values', 'combine_heads'}
    def _label(path, v):
        key    = str(path[-1].key) if hasattr(path[-1], 'key') else ''
        parent = str(path[-2].key) if len(path) > 1 and hasattr(path[-2], 'key') else ''
        if key == 'kernel' and parent in MUON_KEYS:
            return MuonDimensionNumbers(reduction_axis=0, output_axis=1)
        return None
    return jax.tree_util.tree_map_with_path(_label, params)

def is_muon_param(path_str):
    parts = path_str.split('/')
    return (len(parts) >= 2 and
            parts[-1] == 'kernel' and
            parts[-2] in {'to_queries', 'to_keys_values', 'combine_heads'})

def muon_mask(params):
    def _m(path, v):
        return is_muon_param('/'.join(str(p.key) for p in path))
    return jax.tree_util.tree_map_with_path(_m, params)

def is_gate_param(path_str):
    # Проверяем, относится ли параметр к гейту
    return 'memory_gate' in path_str.split('/')

def gate_mask(params):
    def _m(path, v):
        return is_gate_param('/'.join(str(p.key) for p in path))
    return jax.tree_util.tree_map_with_path(_m, params)

def adam_base_mask(params):
    def _m(path, v):
        path_str = '/'.join(str(p.key) for p in path)
        return not is_muon_param(path_str) and not is_gate_param(path_str)
    return jax.tree_util.tree_map_with_path(_m, params)


## lr for experiments 4e-4, 1.5e-4
inner_chain = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.masked(
        optax.contrib.muon(
            learning_rate=lr_muon,
            muon_weight_dimension_numbers=muon_only_dims,
            # beta=0.80,
            eps=1e-8,
            mu_dtype=jnp.float32,
        ),
        mask=muon_mask,
    ),

    # 2. Adam для гейтов (с высоким LR)
    optax.masked(
        adam_atan2(
            learning_rate=lr_gate,
            b1=0.98,  # Еще больше стабильности, так как LR и так мал
            b2=0.95,
            eps=1e-8,
            mu_dtype=jnp.float32
        ),
        mask=gate_mask,
    ),
    # 3. Adam для остальных параметров (с обычным LR)
    optax.masked(
        adam_atan2(
            learning_rate=lr_adam,
                b1=0.95,  # Повышаем для стабильности
                b2=0.90,  # Понижаем, чтобы быстрее адаптироваться
            eps=1e-8,
            mu_dtype=jnp.float32
        ),
        mask=adam_base_mask,
    ),
)

routing_optimizer = optax.MultiSteps(
    kd.optim.partial_updates(
        inner_chain,
        mask=kd.optim.select(["memory", "memory_gate"]),
    ),
    every_k_schedule=16,
)

In [ ]:
print(f"lr_gate {lr_gate(34000)}")
print(f"lr_muon {lr_muon(34000)}")
print(f"lr_adam {lr_adam(34000)}")

## 6. Loss и метрики

In [ ]:
import flax
import flax.linen as nn
from kauldron import metrics as kd_metrics
from kauldron import kontext

class FullParamsInit(kd.ckpts.InitTransform):
    def __init__(self, params):
        self.params = params
    def transform(self, state: kd.train.TrainState) -> kd.train.TrainState:
        return state.replace(params=self.params)

# LM loss — основной сигнал Phase 2
train_losses = {
    "lm_loss": kd.losses.Value(
        values="preds.layer_losses['lm_loss']"
    )
}

# Метрики
@dataclasses.dataclass(kw_only=True, frozen=True, eq=True)
class MemoryGateMetric(kd_metrics.Metric):
    params: kd.kontext.Key = "params"

    @flax.struct.dataclass
    class State(kd_metrics.State):
        gate_metrics: flax.core.FrozenDict[str, jnp.ndarray] = flax.core.FrozenDict()
        def compute(self):
            return {k: np.array(v, dtype=np.float32) for k, v in self.gate_metrics.items()}
        @classmethod
        def empty(cls):
            return cls(gate_metrics=flax.core.FrozenDict())
        def merge(self, other):
            return self

    def get_state(self, params=None, **kwargs) -> State:
        if params is None:
            return self.State.empty()
        stats_dict = {}
        def find_gates(tree, path_prefix=""):
            if hasattr(tree, 'items'):
                for key, val in tree.items():
                    current_path = f"{path_prefix}_{key}" if path_prefix else str(key)
                    if key == "memory_gate":
                        mean_val = jnp.mean(val)
                        openness = jax.nn.sigmoid(mean_val)
                        stats_dict[f"titans_gates/{current_path}_raw_mean"] = mean_val
                        stats_dict[f"titans_gates/{current_path}_openness"] = openness
                        std_val = jnp.std(val)
                        stats_dict[f"titans_gates/{current_path}_raw_std"] = std_val
                    else:
                        find_gates(val, current_path)
        find_gates(params)
        return self.State(gate_metrics=flax.core.freeze(stats_dict))

@flax.struct.dataclass
class LRState(kd_metrics.State):
    lr_value: jnp.ndarray
    @classmethod
    def empty(cls):
        return cls(lr_value=jnp.array(0.0))
    def merge(self, other):
        return self
    def compute(self):
        return self.lr_value

@dataclasses.dataclass(kw_only=True, frozen=True, eq=True)
class HuberDeltaMetric(kd_metrics.Metric):
    step: kontext.Key = "step"
    def get_state(self, step, **kwargs):
        return LRState(lr_value=jnp.array(huber_loss_delta(step)))
@dataclasses.dataclass(kw_only=True, frozen=True, eq=True)
class AdamLearningRateMetric(kd_metrics.Metric):
    step: kontext.Key = "step"
    def get_state(self, step, **kwargs):
        return LRState(lr_value=jnp.array(lr_adam(step)))

class TPUMemoryMetric(kd_metrics.Metric):
    """Метрика для логирования использования памяти TPU в ГБ."""
    @flax.struct.dataclass
    class State(kd_metrics.State):
        # Состояние может быть пустым, так как данные мы берем напрямую из JAX на хосте
        def compute(self):
            stats_dict = {}
            for i, device in enumerate(jax.devices()):
                try:
                    m_stats = device.memory_stats()
                    # Если словарь пустой, пропускаем устройство
                    if not m_stats:
                        continue
                    prefix = f"device_{i}"
                    # 1. Текущее использование памяти (если ключа нет, вернет 0)
                    if 'bytes_in_use' in m_stats:
                        used_gb = m_stats['bytes_in_use'] / 1e9
                        stats_dict[f"{prefix}/used_gb"] = np.array(used_gb, dtype=np.float32)
                    # 2. Пиковое использование
                    if 'peak_bytes_in_use' in m_stats:
                        peak_gb = m_stats['peak_bytes_in_use'] / 1e9
                        stats_dict[f"{prefix}/peak_gb"] = np.array(peak_gb, dtype=np.float32)
                    # 3. Лимит и процент (только если 'limit' действительно существует)
                    if 'limit' in m_stats and 'bytes_in_use' in m_stats:
                        limit_gb = m_stats['limit'] / 1e9
                        usage_pct = (m_stats['bytes_in_use'] / m_stats['limit']) * 100
                        stats_dict[f"{prefix}/usage_pct"] = np.array(usage_pct, dtype=np.float32)
                except (AttributeError, ValueError, RuntimeError):
                    pass
            return stats_dict
        @classmethod
        def empty(cls):
            """Создает пустое начальное состояние."""
            return cls()
        def merge(self, other):
            """Объединяет состояния из разных батчей (здесь ничего не делаем)."""
            return self

    def get_state(self, **kwargs) -> State:
        # Просто возвращаем пустое состояние.
        # Нам не нужны данные из batch или модели для этой метрики.
        return self.State().empty()

@dataclasses.dataclass(kw_only=True, frozen=True, eq=True)
class Perplexity(kd.metrics.Metric):
    # Берет усредненный loss по батчу
    loss: kd.kontext.Key = "preds.layer_losses['lm_loss']"

    @flax.struct.dataclass
    class State(kd.metrics.base_state.AverageState):
        def compute(self):
            # Средний loss за все шаги -> возводим в экспоненту
            mean_loss = super().compute()
            return jnp.exp(mean_loss)
    def get_state(self,*, loss) -> State:
        return self.State.from_values(values=loss)

@dataclasses.dataclass(kw_only=True, frozen=True, eq=True)
class LMAccuracy(kd.metrics.Metric):
    # Берет точность по батчу
    acc: kd.kontext.Key = "preds.layer_losses['lm_accuracy']"
    @flax.struct.dataclass
    class State(kd.metrics.base_state.AverageState):
        pass
    def get_state(self,*, acc) -> State:
        return self.State.from_values(values=acc)


train_metrics = {
    "LM/accuracy": LMAccuracy(),
    "LM/perplexity": Perplexity(),
    "LM/adam_lr": AdamLearningRateMetric(),
    "memory_gate": MemoryGateMetric(),
    "tpu_memory": TPUMemoryMetric()
}
train_summaries = {}
for layer in active_titans_layers:
    key = f"Gates_Dist_{layer}"
    train_summaries[key] = kd.summaries.HistogramSummary(tensor=f"params.layer_{layer}.memory_gate")


## 3. Модель (training_phase=2)

In [ ]:
# !git pull

In [ ]:
# # import gemma_titans
# import gemma_titans
# importlib.reload(gemma_titans)
# from gemma_titans import Gemma3_1B_Titans, Gemma_Titans_Config
# from titans_ckpts import SkipTitans

In [ ]:
# gemma_config = dataclasses.replace(
#     Gemma3_1B_Titans.config,
#     sliding_window_size=128,
#     training_phase=2,
#     titans_layer_indices=active_titans_layers,
#     titans_phase2_first_layer=titans_phase2_first_layer,
#     neural_mem_huber_delta=huber_loss_delta,
# )


# model = Gemma3_1B_Titans(
#     config=gemma_config,
#     dtype=jnp.bfloat16,
#     return_last_only=False,
#     tokens="batch.tokens",
# )

# print(f"Model: titans_layer_indices={gemma_config.titans_layer_indices}, phase2_first={gemma_config.titans_phase2_first_layer}")

## 7. Trainer

In [ ]:
workdir_name = f'titans_workdir_phase2_from{titans_phase2_first_layer}'


trainer = kd.train.Trainer(
    seed=42,
    workdir=os.path.abspath(f'./{workdir_name}'),
    train_ds=get_c4_webtextlike_dataset(
        split="train",
        batch_size=batch_size,
        tokenizer=gm.text.Gemma3Tokenizer(), # pass tokenizer instance here
        shuffle=True,
        num_epochs=None
    ),
    model=model,
    # init_transform=FullParamsInit(merged_params) if 'merged_params' in dir() else None,
    num_train_steps=total_steps,
    train_losses=train_losses,
    train_metrics=train_metrics,
    train_summaries = train_summaries,
    optimizer=routing_optimizer,
    checkpointer=kd.ckpts.Checkpointer(save_interval_steps=500),
)

print(f"Trainer initialized. workdir: {workdir_name}")

## 8. TensorBoard

Если у вас уже запущено длительное обучение trainer.train() в ячейке, и нужно запустить заново Tensorboard, вам нужно воспользоваться Терминалом Colab:

terminal command available for runtime restart:
```
fuser -k 6006/tcp && tensorboard --logdir /content/Titans_jax/titans_workdir_phase2_from11/ --port 6006 &
```
после чего обновить Tensorboard


In [ ]:
%reload_ext tensorboard
from tensorboard import notebook

# Показать список всех активных инстансов
notebook.list()



In [ ]:
!rm -rf /tmp/.tensorboard-info/*

In [ ]:
%tensorboard --logdir ./{workdir_name}/ --port=6006

## 9. Обучение

In [ ]:
state, aux = trainer.train()


In [ ]:
import jax
import gc

# 1. DESTROY the Python references to the old TPU buffers FIRST
try:
    del state
    del aux
except NameError:
    pass

# 2. Clear JAX's internal live buffer cache
for device in jax.devices():
    if hasattr(device, 'live_arrays'): device.live_arrays().clear()
    if hasattr(device, 'live_buffers'): device.live_buffers().clear()
    if hasattr(device, 'default_memory_tracker'): device.default_memory_tracker().clear()

jax.clear_caches()

# 3. NOW run Python garbage collection
gc.collect()

print("TPU memory cache cleared and garbage collection finished.")


In [ ]:

trainer = trainer.replace(num_train_steps=85000)

state, aux = trainer.train()

## 10. Сохранение весов

In [ ]:
def save_titans_weights(state: kd.train.TrainState, save_dir: str):
    _, titans_params = titans_tree_utils.split_titans_params(state.params)
    save_path = os.path.abspath(save_dir)
    if os.path.exists(save_path):
        shutil.rmtree(save_path)
    checkpointer = ocp.StandardCheckpointer()
    checkpointer.save(save_path, titans_params)
    checkpointer.wait_until_finished()
    print(f"Saved Titans weights to {save_path}")
new_weights_name = f"saved_titans_phase2_from_{titans_phase2_first_layer}"

save_titans_weights(state, f"./{new_weights_name}")

shutil.make_archive(new_weights_name, 'zip', f"./{new_weights_name}")
print(f"\u2705 {new_weights_name}.zip ready")
destination_path = f"/content/drive/Shareddrives/shared_veriga/saved_titans_delta/{new_weights_name}.zip"
shutil.copy(f"./{new_weights_name}.zip", destination_path)
print(f"Файл скопирован в: {destination_path}")